# Maze Crawler Worker Wall-Removal Agent

Experimental Kaggle notebook that keeps the Version 6 jump-BFS factory/scout core and adds one conservative worker for route opening.


## 1. Algorithm Summary

This notebook starts from the Version 6 jump-BFS reference agent and tests a separate worker wall-removal strategy. The key ideas are:

1. Keep jump-preferred factory BFS, remembered walls, mirrored wall inference, and one active replacement scout.
2. Build at most one worker only after the factory is safely ahead of `southBound`.
3. Prefer scout vision before worker economy: the worker is not built until an active scout exists.
4. Route the worker toward the cells ahead of the factory using known-wall BFS.
5. Use `REMOVE_NORTH` only when the worker has enough energy and a known north wall blocks progress.
6. Leave miners out of this experiment so worker value can be measured cleanly.

This is intentionally a separate notebook because workers add new costs, collision risks, and wall-editing actions. Version 6 remains the reference champion until this worker line proves stronger on leaderboard episodes.


## 2. Setup

The simulation cells need Kaggle Environments with the `crawl` environment available. Kaggle usually has the package ready, but this cell installs or upgrades it if needed. The next cell centralizes tunable run settings and the replay helper used by both simulations.


In [ ]:
%%capture
!pip install -q "kaggle-environments>=1.29.0"


In [ ]:
from kaggle_environments import make

RANDOM_SEED = 42
RENDER_WIDTH = 800
RENDER_HEIGHT = 800
RUN_SIMULATIONS = True
RUN_BATCH_EVALUATION = False
EVAL_SEEDS = 20
USE_FACTORY_DANGER_GATE = False


def run_simulation(agent_fn, label: str):
    """Run one match against a random opponent and render the replay."""
    if not RUN_SIMULATIONS:
        print(f"{label}: simulation skipped")
        return None

    env = make("crawl", configuration={"randomSeed": RANDOM_SEED}, debug=True)
    env.run([agent_fn, "random"])

    print(label)
    for player_id, step in enumerate(env.steps[-1]):
        print(
            f"Player {player_id}: reward={step.reward}, "
            f"status={step.status}"
        )

    return env.render(
        mode="ipython",
        width=RENDER_WIDTH,
        height=RENDER_HEIGHT,
    )


print("kaggle_environments ready")


## 3. Agent Helpers

These helpers normalize Kaggle objects and dictionaries, cache visible wall information across turns, and mirror known walls across the symmetric board. Unknown cells remain optimistic, which keeps the starter compact while giving BFS more memory than a single observation.


In [ ]:
from collections import deque
from typing import Any

FACTORY = 0
SCOUT = 1
WORKER = 2

DIRS = {
    "NORTH": (0, 1),
    "SOUTH": (0, -1),
    "EAST": (1, 0),
    "WEST": (-1, 0),
}
MOVE_ORDER = ("NORTH", "EAST", "WEST", "SOUTH")
WALL_BIT = {
    "NORTH": 1,
    "EAST": 2,
    "SOUTH": 4,
    "WEST": 8,
}
MIRROR_WALL = [0] * 16
for value in range(16):
    mirrored = value & (WALL_BIT["NORTH"] | WALL_BIT["SOUTH"])
    if value & WALL_BIT["EAST"]:
        mirrored |= WALL_BIT["WEST"]
    if value & WALL_BIT["WEST"]:
        mirrored |= WALL_BIT["EAST"]
    MIRROR_WALL[value] = mirrored

_wall_memory: dict[tuple[int, int], int] = {}


def get_value(obj: Any, key: str, default: Any = None) -> Any:
    """Return a field from either a dict-like or object-like value.

    Args:
        obj: Observation/config object from Kaggle or a plain dict.
        key: Field name to read.
        default: Fallback when the field is missing.

    Returns:
        The requested value or `default`.
    """
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def update_wall_memory(obs: Any, config: Any) -> None:
    """Cache visible walls and mirrored walls for later pathfinding."""
    walls = get_value(obs, "walls", [])
    south_bound = get_value(obs, "southBound", 0)
    width = get_value(config, "width", 0)

    for index, wall_value in enumerate(walls):
        if wall_value == -1:
            continue

        row = south_bound + index // width
        col = index % width
        wall_bits = int(wall_value)
        _wall_memory[(col, row)] = wall_bits

        mirror_col = width - 1 - col
        _wall_memory.setdefault((mirror_col, row), MIRROR_WALL[wall_bits])

    if len(_wall_memory) > 2500:
        cutoff = south_bound - 5
        old_cells = [cell for cell in _wall_memory if cell[1] < cutoff]
        for cell in old_cells:
            del _wall_memory[cell]


def wall_at(obs: Any, config: Any, col: int, row: int) -> int:
    """Return the known wall bitfield at a cell.

    Args:
        obs: Current Maze Crawler observation.
        config: Environment configuration.
        col: Cell column.
        row: Cell row.

    Returns:
        Wall bitfield. Undiscovered cells are treated as open in this starter.
    """
    if (col, row) in _wall_memory:
        return _wall_memory[(col, row)]

    walls = get_value(obs, "walls", [])
    south_bound = get_value(obs, "southBound", 0)
    width = get_value(config, "width", 0)
    index = (row - south_bound) * width + col

    if 0 <= index < len(walls) and walls[index] != -1:
        return int(walls[index])
    return 0


def in_bounds(obs: Any, config: Any, col: int, row: int) -> bool:
    """Return True when a cell is inside the currently playable board."""
    width = get_value(config, "width", 0)
    south_bound = get_value(obs, "southBound", 0)
    north_bound = get_value(obs, "northBound", south_bound + 50)
    return 0 <= col < width and south_bound <= row <= north_bound


def has_road(
    obs: Any,
    config: Any,
    col: int,
    row: int,
    direction: str,
) -> bool:
    """Return True when no known wall blocks a direction from a cell."""
    next_col, next_row = neighbor(col, row, direction)
    if not in_bounds(obs, config, next_col, next_row):
        return False
    return not (wall_at(obs, config, col, row) & WALL_BIT[direction])


def can_jump(
    obs: Any,
    config: Any,
    col: int,
    row: int,
    direction: str,
) -> bool:
    """Return True when a factory jump can land in a plausible cell."""
    delta_col, delta_row = DIRS[direction]
    next_col = col + 2 * delta_col
    next_row = row + 2 * delta_row
    if not in_bounds(obs, config, next_col, next_row):
        return False
    return wall_at(obs, config, next_col, next_row) != 15


def neighbor(col: int, row: int, direction: str) -> tuple[int, int]:
    """Return the adjacent cell in `direction`."""
    delta_col, delta_row = DIRS[direction]
    return col + delta_col, row + delta_row


def jump_neighbor(col: int, row: int, direction: str) -> tuple[int, int]:
    """Return the factory jump destination in `direction`."""
    delta_col, delta_row = DIRS[direction]
    return col + 2 * delta_col, row + 2 * delta_row


def my_robots(obs: Any) -> dict[str, list[int]]:
    """Return robots controlled by the current player."""
    player = get_value(obs, "player", 0)
    robots = get_value(obs, "robots", {}) or {}
    return {
        uid: data
        for uid, data in robots.items()
        if len(data) > 4 and data[4] == player
    }


def my_factory(obs: Any) -> tuple[str | None, list[int] | None]:
    """Return our factory id and data, if visible."""
    for uid, data in my_robots(obs).items():
        if data[0] == FACTORY:
            return uid, data
    return None, None


def occupied_positions(robots: dict[str, list[int]]) -> set[tuple[int, int]]:
    """Return currently occupied cells for our robots."""
    return {(data[1], data[2]) for data in robots.values()}


def parse_cell(key: str) -> tuple[int, int]:
    """Parse a `col,row` map key into integer coordinates."""
    col, row = key.split(",")
    return int(col), int(row)


## 4. Factory Jump-Preferred BFS

The factory now uses a lightweight version of the stronger notebook's main idea: cache wall knowledge, plan toward a higher row with BFS, and explore jump moves before normal walks when the jump cooldown is ready. If the factory is close to the scroll boundary, it switches to emergency escape rules instead of idling in a local dead end.


In [ ]:
def bfs_first_step(
    start: tuple[int, int],
    goals: list[tuple[int, int]],
    obs: Any,
    config: Any,
    depth: int = 20,
    occupied: set[tuple[int, int]] | None = None,
    avoid_occupied: bool = True,
) -> str | None:
    """Return the first walking step to any goal using known open roads."""
    if not goals:
        return None

    occupied = occupied or set()
    goal_set = set(goals)
    if start in goal_set:
        return "IDLE"

    queue = deque([(start, None, 0)])
    seen = {start}

    while queue:
        (col, row), first_step, distance = queue.popleft()
        if (col, row) in goal_set and distance > 0:
            return first_step
        if distance >= depth:
            continue

        for direction in MOVE_ORDER:
            if not has_road(obs, config, col, row, direction):
                continue
            next_cell = neighbor(col, row, direction)
            if next_cell in seen:
                continue
            if avoid_occupied and next_cell in occupied and next_cell != start:
                continue
            seen.add(next_cell)
            queue.append((next_cell, first_step or direction, distance + 1))

    return None


def bfs_jump_preferred(
    start: tuple[int, int],
    goals: list[tuple[int, int]],
    jump_cd: int,
    obs: Any,
    config: Any,
    depth: int = 20,
    occupied: set[tuple[int, int]] | None = None,
) -> str | None:
    """Return a factory step, searching jumps before walks when possible."""
    if not goals:
        return None

    occupied = occupied or set()
    goal_set = set(goals)
    queue = deque([(start, None, 0, max(0, jump_cd))])
    seen = {(start[0], start[1], jump_cd <= 0)}

    while queue:
        (col, row), first_step, distance, current_cd = queue.popleft()
        if (col, row) in goal_set and distance > 0:
            return first_step
        if distance >= depth:
            continue

        if current_cd <= 0:
            for direction in MOVE_ORDER:
                if not can_jump(obs, config, col, row, direction):
                    continue
                next_cell = jump_neighbor(col, row, direction)
                if next_cell in occupied and next_cell != start:
                    continue
                key = (next_cell[0], next_cell[1], False)
                if key in seen:
                    continue
                seen.add(key)
                queue.append((
                    next_cell,
                    first_step or f"JUMP_{direction}",
                    distance + 1,
                    20,
                ))

        for direction in MOVE_ORDER:
            if not has_road(obs, config, col, row, direction):
                continue
            next_cell = neighbor(col, row, direction)
            if next_cell in occupied and next_cell != start:
                continue
            next_cd = max(0, current_cd - 1)
            key = (next_cell[0], next_cell[1], next_cd <= 0)
            if key in seen:
                continue
            seen.add(key)
            queue.append((
                next_cell,
                first_step or direction,
                distance + 1,
                next_cd,
            ))

    return None


def factory_path_action(
    col: int,
    row: int,
    move_cd: int,
    jump_cd: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str:
    """Choose a survival-focused factory action with BFS fallbacks."""
    occupied = occupied or set()
    south_bound = get_value(obs, "southBound", 0)
    north_bound = get_value(obs, "northBound", row + 50)
    width = get_value(config, "width", 0)
    factory_gap = row - south_bound

    if move_cd > 1:
        return "IDLE"

    if factory_gap <= 2 and jump_cd == 0:
        for direction in ("NORTH", "EAST", "WEST"):
            landing = jump_neighbor(col, row, direction)
            if can_jump(obs, config, col, row, direction):
                if landing not in occupied:
                    return f"JUMP_{direction}"

    target_row = min(north_bound, row + 20)
    goals = [(target_col, target_row) for target_col in range(width)]
    step = bfs_jump_preferred(
        (col, row),
        goals,
        jump_cd,
        obs,
        config,
        depth=22,
        occupied=occupied,
    )

    if step is None:
        closer_row = min(north_bound, row + 6)
        closer_goals = [(target_col, closer_row) for target_col in range(width)]
        step = bfs_first_step(
            (col, row),
            closer_goals,
            obs,
            config,
            depth=12,
            occupied=occupied,
            avoid_occupied=False,
        )

    if step is None:
        for direction in ("NORTH", "EAST", "WEST"):
            if has_road(obs, config, col, row, direction):
                next_cell = neighbor(col, row, direction)
                if next_cell not in occupied:
                    return direction

    if step is None and factory_gap <= 3:
        if has_road(obs, config, col, row, "SOUTH"):
            next_cell = neighbor(col, row, "SOUTH")
            if next_cell not in occupied:
                return "SOUTH"

    if step is None and factory_gap <= 3 and jump_cd == 0:
        for direction in ("EAST", "WEST", "SOUTH"):
            landing = jump_neighbor(col, row, direction)
            if can_jump(obs, config, col, row, direction):
                if landing not in occupied:
                    return f"JUMP_{direction}"

    return step or "IDLE"


def agent_v1(obs: Any, config: Any) -> dict[str, str]:
    """Factory-only baseline for a quick survival simulation."""
    update_wall_memory(obs, config)
    actions = {}
    robots = my_robots(obs)
    occupied = occupied_positions(robots)

    for uid, data in robots.items():
        robot_type, col, row = data[0], data[1], data[2]
        move_cd = data[5] if len(data) > 5 else 0
        jump_cd = data[6] if len(data) > 6 else 0
        if robot_type == FACTORY:
            occupied_without_self = occupied - {(col, row)}
            actions[uid] = factory_path_action(
                col,
                row,
                move_cd,
                jump_cd,
                obs,
                config,
                occupied_without_self,
            )
    return actions


## 5. Agent V1 Simulation

Run the factory-only baseline against a random opponent and render the replay. This is the quick sanity check that `crawl` is installed and the basic action dictionary is valid.


In [ ]:
run_simulation(agent_v1, "Agent V1: factory-only baseline")


## 6. Scout And Worker Support

The support policy keeps one active scout for vision and adds at most one conservative worker. The factory builds scouts first, then considers a worker only when it has enough energy and enough distance from the scroll boundary.


In [ ]:
SCOUT_RETURN_ENERGY = 75
DEFAULT_SCOUT_COST = 50
DEFAULT_WORKER_COST = 200
DEFAULT_WALL_REMOVE_COST = 100
MAX_ACTIVE_SCOUTS = 1
MAX_ACTIVE_WORKERS = 1
WORKER_BUILD_GAP = 8
WORKER_FRONT_OFFSET = 2
WORKER_MIN_FACTORY_ENERGY = 650
USE_FACTORY_DANGER_GATE = False
FACTORY_DANGER_GAP = 4
_last_south_bound: int | None = None


def reset_game_state_if_needed(obs: Any) -> None:
    """Reset persistent state when Kaggle starts a new episode."""
    global _last_south_bound

    south_bound = get_value(obs, "southBound", 0)
    if _last_south_bound is not None and south_bound < _last_south_bound:
        _wall_memory.clear()

    _last_south_bound = south_bound


def scout_action(
    col: int,
    row: int,
    energy: int,
    factory_pos: tuple[int, int],
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str:
    """Collect visible crystals and ferry high energy back to the factory."""
    occupied = occupied or set()
    factory_col, factory_row = factory_pos
    if energy >= SCOUT_RETURN_ENERGY:
        for direction in MOVE_ORDER:
            is_adjacent = neighbor(col, row, direction) == (
                factory_col,
                factory_row,
            )
            if is_adjacent and has_road(obs, config, col, row, direction):
                return f"TRANSFER_{direction}"

        step = bfs_first_step(
            (col, row),
            [factory_pos],
            obs,
            config,
            depth=16,
            occupied=occupied,
            avoid_occupied=False,
        )
        return step or "IDLE"

    crystals = [
        parse_cell(key)
        for key in (get_value(obs, "crystals", {}) or {})
    ]
    if crystals:
        targets = sorted(
            crystals,
            key=lambda cell: abs(cell[0] - col) + abs(cell[1] - row),
        )[:3]
    else:
        targets = [(factory_col, factory_row + 8)]

    step = bfs_first_step(
        (col, row),
        targets,
        obs,
        config,
        depth=16,
        occupied=occupied,
    )
    if step and step != "IDLE":
        return step

    for direction in ("NORTH", "EAST", "WEST"):
        if has_road(obs, config, col, row, direction):
            next_cell = neighbor(col, row, direction)
            if next_cell not in occupied:
                return direction
    return "IDLE"


def worker_action(
    col: int,
    row: int,
    energy: int,
    factory_pos: tuple[int, int],
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str:
    """Open a route ahead of the factory without adding broad economy logic."""
    occupied = occupied or set()
    remove_cost = get_value(
        config,
        "wallRemoveCost",
        DEFAULT_WALL_REMOVE_COST,
    )
    if energy >= remove_cost and not has_road(obs, config, col, row, "NORTH"):
        return "REMOVE_NORTH"

    factory_col, factory_row = factory_pos
    target = (factory_col, factory_row + WORKER_FRONT_OFFSET)
    step = bfs_first_step(
        (col, row),
        [target],
        obs,
        config,
        depth=10,
        occupied=occupied,
    )
    if step and step != "IDLE":
        return step

    for direction in ("NORTH", "EAST", "WEST"):
        if has_road(obs, config, col, row, direction):
            next_cell = neighbor(col, row, direction)
            if next_cell not in occupied:
                return direction
    return "IDLE"


def can_spawn_scout(
    col: int,
    row: int,
    energy: int,
    build_cd: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]],
) -> bool:
    """Return True when the factory can safely spawn one scout north."""
    scout_cost = get_value(config, "scoutCost", DEFAULT_SCOUT_COST)
    spawn = neighbor(col, row, "NORTH")
    south_bound = get_value(obs, "southBound", 0)
    factory_gap = row - south_bound
    danger_gate_ok = (
        not USE_FACTORY_DANGER_GATE
        or factory_gap > FACTORY_DANGER_GAP
    )
    return (
        danger_gate_ok
        and build_cd == 0
        and energy >= scout_cost
        and spawn not in occupied
        and has_road(obs, config, col, row, "NORTH")
    )


def can_spawn_worker(
    col: int,
    row: int,
    energy: int,
    build_cd: int,
    scout_count: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]],
) -> bool:
    """Return True when the worker experiment can safely spend energy."""
    worker_cost = get_value(config, "workerCost", DEFAULT_WORKER_COST)
    spawn = neighbor(col, row, "NORTH")
    south_bound = get_value(obs, "southBound", 0)
    factory_gap = row - south_bound
    return (
        scout_count > 0
        and factory_gap > WORKER_BUILD_GAP
        and build_cd == 0
        and energy >= max(worker_cost, WORKER_MIN_FACTORY_ENERGY)
        and spawn not in occupied
        and has_road(obs, config, col, row, "NORTH")
    )


def agent_v2(obs: Any, config: Any) -> dict[str, str]:
    """Jump-BFS reference with one scout and one conservative worker."""
    reset_game_state_if_needed(obs)
    update_wall_memory(obs, config)
    actions = {}
    robots = my_robots(obs)
    _, factory_data = my_factory(obs)
    occupied = occupied_positions(robots)
    scout_count = sum(1 for data in robots.values() if data[0] == SCOUT)
    worker_count = sum(1 for data in robots.values() if data[0] == WORKER)

    for uid, data in robots.items():
        robot_type, col, row, energy = data[0], data[1], data[2], data[3]
        move_cd = data[5] if len(data) > 5 else 0
        jump_cd = data[6] if len(data) > 6 else 0
        build_cd = data[7] if len(data) > 7 else 0
        occupied_without_self = occupied - {(col, row)}

        if robot_type == FACTORY:
            if scout_count < MAX_ACTIVE_SCOUTS and can_spawn_scout(
                col,
                row,
                energy,
                build_cd,
                obs,
                config,
                occupied_without_self,
            ):
                actions[uid] = "BUILD_SCOUT"
                scout_count += 1
            elif worker_count < MAX_ACTIVE_WORKERS and can_spawn_worker(
                col,
                row,
                energy,
                build_cd,
                scout_count,
                obs,
                config,
                occupied_without_self,
            ):
                actions[uid] = "BUILD_WORKER"
                worker_count += 1
            else:
                actions[uid] = factory_path_action(
                    col,
                    row,
                    move_cd,
                    jump_cd,
                    obs,
                    config,
                    occupied_without_self,
                )
        elif robot_type == SCOUT and factory_data is not None:
            factory_pos = (factory_data[1], factory_data[2])
            actions[uid] = scout_action(
                col,
                row,
                energy,
                factory_pos,
                obs,
                config,
                occupied_without_self,
            )
        elif robot_type == WORKER and factory_data is not None:
            factory_pos = (factory_data[1], factory_data[2])
            actions[uid] = worker_action(
                col,
                row,
                energy,
                factory_pos,
                obs,
                config,
                occupied_without_self,
            )

    return actions


## 7. Agent V2 Worker Simulation


In [ ]:
_wall_memory.clear()
_last_south_bound = None
run_simulation(agent_v2, "Agent V2: jump BFS plus scout and worker")


## 8. Generate Submission Files

Kaggle imports `agent` from `main.py`. This cell writes the final agent source to `main.py`; the next cell mirrors that exact source to `submission.py` for tooling that expects the alternate filename. Keep generated files out of git.


In [ ]:
%%writefile main.py
from collections import deque
from typing import Any

FACTORY = 0
SCOUT = 1
WORKER = 2

DIRS = {
    "NORTH": (0, 1),
    "SOUTH": (0, -1),
    "EAST": (1, 0),
    "WEST": (-1, 0),
}
MOVE_ORDER = ("NORTH", "EAST", "WEST", "SOUTH")
WALL_BIT = {
    "NORTH": 1,
    "EAST": 2,
    "SOUTH": 4,
    "WEST": 8,
}
MIRROR_WALL = [0] * 16
for value in range(16):
    mirrored = value & (WALL_BIT["NORTH"] | WALL_BIT["SOUTH"])
    if value & WALL_BIT["EAST"]:
        mirrored |= WALL_BIT["WEST"]
    if value & WALL_BIT["WEST"]:
        mirrored |= WALL_BIT["EAST"]
    MIRROR_WALL[value] = mirrored

_wall_memory: dict[tuple[int, int], int] = {}


def get_value(obj: Any, key: str, default: Any = None) -> Any:
    """Return a field from either a dict-like or object-like value.

    Args:
        obj: Observation/config object from Kaggle or a plain dict.
        key: Field name to read.
        default: Fallback when the field is missing.

    Returns:
        The requested value or `default`.
    """
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def update_wall_memory(obs: Any, config: Any) -> None:
    """Cache visible walls and mirrored walls for later pathfinding."""
    walls = get_value(obs, "walls", [])
    south_bound = get_value(obs, "southBound", 0)
    width = get_value(config, "width", 0)

    for index, wall_value in enumerate(walls):
        if wall_value == -1:
            continue

        row = south_bound + index // width
        col = index % width
        wall_bits = int(wall_value)
        _wall_memory[(col, row)] = wall_bits

        mirror_col = width - 1 - col
        _wall_memory.setdefault((mirror_col, row), MIRROR_WALL[wall_bits])

    if len(_wall_memory) > 2500:
        cutoff = south_bound - 5
        old_cells = [cell for cell in _wall_memory if cell[1] < cutoff]
        for cell in old_cells:
            del _wall_memory[cell]


def wall_at(obs: Any, config: Any, col: int, row: int) -> int:
    """Return the known wall bitfield at a cell.

    Args:
        obs: Current Maze Crawler observation.
        config: Environment configuration.
        col: Cell column.
        row: Cell row.

    Returns:
        Wall bitfield. Undiscovered cells are treated as open in this starter.
    """
    if (col, row) in _wall_memory:
        return _wall_memory[(col, row)]

    walls = get_value(obs, "walls", [])
    south_bound = get_value(obs, "southBound", 0)
    width = get_value(config, "width", 0)
    index = (row - south_bound) * width + col

    if 0 <= index < len(walls) and walls[index] != -1:
        return int(walls[index])
    return 0


def in_bounds(obs: Any, config: Any, col: int, row: int) -> bool:
    """Return True when a cell is inside the currently playable board."""
    width = get_value(config, "width", 0)
    south_bound = get_value(obs, "southBound", 0)
    north_bound = get_value(obs, "northBound", south_bound + 50)
    return 0 <= col < width and south_bound <= row <= north_bound


def has_road(
    obs: Any,
    config: Any,
    col: int,
    row: int,
    direction: str,
) -> bool:
    """Return True when no known wall blocks a direction from a cell."""
    next_col, next_row = neighbor(col, row, direction)
    if not in_bounds(obs, config, next_col, next_row):
        return False
    return not (wall_at(obs, config, col, row) & WALL_BIT[direction])


def can_jump(
    obs: Any,
    config: Any,
    col: int,
    row: int,
    direction: str,
) -> bool:
    """Return True when a factory jump can land in a plausible cell."""
    delta_col, delta_row = DIRS[direction]
    next_col = col + 2 * delta_col
    next_row = row + 2 * delta_row
    if not in_bounds(obs, config, next_col, next_row):
        return False
    return wall_at(obs, config, next_col, next_row) != 15


def neighbor(col: int, row: int, direction: str) -> tuple[int, int]:
    """Return the adjacent cell in `direction`."""
    delta_col, delta_row = DIRS[direction]
    return col + delta_col, row + delta_row


def jump_neighbor(col: int, row: int, direction: str) -> tuple[int, int]:
    """Return the factory jump destination in `direction`."""
    delta_col, delta_row = DIRS[direction]
    return col + 2 * delta_col, row + 2 * delta_row


def my_robots(obs: Any) -> dict[str, list[int]]:
    """Return robots controlled by the current player."""
    player = get_value(obs, "player", 0)
    robots = get_value(obs, "robots", {}) or {}
    return {
        uid: data
        for uid, data in robots.items()
        if len(data) > 4 and data[4] == player
    }


def my_factory(obs: Any) -> tuple[str | None, list[int] | None]:
    """Return our factory id and data, if visible."""
    for uid, data in my_robots(obs).items():
        if data[0] == FACTORY:
            return uid, data
    return None, None


def occupied_positions(robots: dict[str, list[int]]) -> set[tuple[int, int]]:
    """Return currently occupied cells for our robots."""
    return {(data[1], data[2]) for data in robots.values()}


def parse_cell(key: str) -> tuple[int, int]:
    """Parse a `col,row` map key into integer coordinates."""
    col, row = key.split(",")
    return int(col), int(row)

def bfs_first_step(
    start: tuple[int, int],
    goals: list[tuple[int, int]],
    obs: Any,
    config: Any,
    depth: int = 20,
    occupied: set[tuple[int, int]] | None = None,
    avoid_occupied: bool = True,
) -> str | None:
    """Return the first walking step to any goal using known open roads."""
    if not goals:
        return None

    occupied = occupied or set()
    goal_set = set(goals)
    if start in goal_set:
        return "IDLE"

    queue = deque([(start, None, 0)])
    seen = {start}

    while queue:
        (col, row), first_step, distance = queue.popleft()
        if (col, row) in goal_set and distance > 0:
            return first_step
        if distance >= depth:
            continue

        for direction in MOVE_ORDER:
            if not has_road(obs, config, col, row, direction):
                continue
            next_cell = neighbor(col, row, direction)
            if next_cell in seen:
                continue
            if avoid_occupied and next_cell in occupied and next_cell != start:
                continue
            seen.add(next_cell)
            queue.append((next_cell, first_step or direction, distance + 1))

    return None


def bfs_jump_preferred(
    start: tuple[int, int],
    goals: list[tuple[int, int]],
    jump_cd: int,
    obs: Any,
    config: Any,
    depth: int = 20,
    occupied: set[tuple[int, int]] | None = None,
) -> str | None:
    """Return a factory step, searching jumps before walks when possible."""
    if not goals:
        return None

    occupied = occupied or set()
    goal_set = set(goals)
    queue = deque([(start, None, 0, max(0, jump_cd))])
    seen = {(start[0], start[1], jump_cd <= 0)}

    while queue:
        (col, row), first_step, distance, current_cd = queue.popleft()
        if (col, row) in goal_set and distance > 0:
            return first_step
        if distance >= depth:
            continue

        if current_cd <= 0:
            for direction in MOVE_ORDER:
                if not can_jump(obs, config, col, row, direction):
                    continue
                next_cell = jump_neighbor(col, row, direction)
                if next_cell in occupied and next_cell != start:
                    continue
                key = (next_cell[0], next_cell[1], False)
                if key in seen:
                    continue
                seen.add(key)
                queue.append((
                    next_cell,
                    first_step or f"JUMP_{direction}",
                    distance + 1,
                    20,
                ))

        for direction in MOVE_ORDER:
            if not has_road(obs, config, col, row, direction):
                continue
            next_cell = neighbor(col, row, direction)
            if next_cell in occupied and next_cell != start:
                continue
            next_cd = max(0, current_cd - 1)
            key = (next_cell[0], next_cell[1], next_cd <= 0)
            if key in seen:
                continue
            seen.add(key)
            queue.append((
                next_cell,
                first_step or direction,
                distance + 1,
                next_cd,
            ))

    return None


def factory_path_action(
    col: int,
    row: int,
    move_cd: int,
    jump_cd: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str:
    """Choose a survival-focused factory action with BFS fallbacks."""
    occupied = occupied or set()
    south_bound = get_value(obs, "southBound", 0)
    north_bound = get_value(obs, "northBound", row + 50)
    width = get_value(config, "width", 0)
    factory_gap = row - south_bound

    if move_cd > 1:
        return "IDLE"

    if factory_gap <= 2 and jump_cd == 0:
        for direction in ("NORTH", "EAST", "WEST"):
            landing = jump_neighbor(col, row, direction)
            if can_jump(obs, config, col, row, direction):
                if landing not in occupied:
                    return f"JUMP_{direction}"

    target_row = min(north_bound, row + 20)
    goals = [(target_col, target_row) for target_col in range(width)]
    step = bfs_jump_preferred(
        (col, row),
        goals,
        jump_cd,
        obs,
        config,
        depth=22,
        occupied=occupied,
    )

    if step is None:
        closer_row = min(north_bound, row + 6)
        closer_goals = [(target_col, closer_row) for target_col in range(width)]
        step = bfs_first_step(
            (col, row),
            closer_goals,
            obs,
            config,
            depth=12,
            occupied=occupied,
            avoid_occupied=False,
        )

    if step is None:
        for direction in ("NORTH", "EAST", "WEST"):
            if has_road(obs, config, col, row, direction):
                next_cell = neighbor(col, row, direction)
                if next_cell not in occupied:
                    return direction

    if step is None and factory_gap <= 3:
        if has_road(obs, config, col, row, "SOUTH"):
            next_cell = neighbor(col, row, "SOUTH")
            if next_cell not in occupied:
                return "SOUTH"

    if step is None and factory_gap <= 3 and jump_cd == 0:
        for direction in ("EAST", "WEST", "SOUTH"):
            landing = jump_neighbor(col, row, direction)
            if can_jump(obs, config, col, row, direction):
                if landing not in occupied:
                    return f"JUMP_{direction}"

    return step or "IDLE"


def agent_v1(obs: Any, config: Any) -> dict[str, str]:
    """Factory-only baseline for a quick survival simulation."""
    update_wall_memory(obs, config)
    actions = {}
    robots = my_robots(obs)
    occupied = occupied_positions(robots)

    for uid, data in robots.items():
        robot_type, col, row = data[0], data[1], data[2]
        move_cd = data[5] if len(data) > 5 else 0
        jump_cd = data[6] if len(data) > 6 else 0
        if robot_type == FACTORY:
            occupied_without_self = occupied - {(col, row)}
            actions[uid] = factory_path_action(
                col,
                row,
                move_cd,
                jump_cd,
                obs,
                config,
                occupied_without_self,
            )
    return actions

SCOUT_RETURN_ENERGY = 75
DEFAULT_SCOUT_COST = 50
DEFAULT_WORKER_COST = 200
DEFAULT_WALL_REMOVE_COST = 100
MAX_ACTIVE_SCOUTS = 1
MAX_ACTIVE_WORKERS = 1
WORKER_BUILD_GAP = 8
WORKER_FRONT_OFFSET = 2
WORKER_MIN_FACTORY_ENERGY = 650
USE_FACTORY_DANGER_GATE = False
FACTORY_DANGER_GAP = 4
_last_south_bound: int | None = None


def reset_game_state_if_needed(obs: Any) -> None:
    """Reset persistent state when Kaggle starts a new episode."""
    global _last_south_bound

    south_bound = get_value(obs, "southBound", 0)
    if _last_south_bound is not None and south_bound < _last_south_bound:
        _wall_memory.clear()

    _last_south_bound = south_bound


def scout_action(
    col: int,
    row: int,
    energy: int,
    factory_pos: tuple[int, int],
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str:
    """Collect visible crystals and ferry high energy back to the factory."""
    occupied = occupied or set()
    factory_col, factory_row = factory_pos
    if energy >= SCOUT_RETURN_ENERGY:
        for direction in MOVE_ORDER:
            is_adjacent = neighbor(col, row, direction) == (
                factory_col,
                factory_row,
            )
            if is_adjacent and has_road(obs, config, col, row, direction):
                return f"TRANSFER_{direction}"

        step = bfs_first_step(
            (col, row),
            [factory_pos],
            obs,
            config,
            depth=16,
            occupied=occupied,
            avoid_occupied=False,
        )
        return step or "IDLE"

    crystals = [
        parse_cell(key)
        for key in (get_value(obs, "crystals", {}) or {})
    ]
    if crystals:
        targets = sorted(
            crystals,
            key=lambda cell: abs(cell[0] - col) + abs(cell[1] - row),
        )[:3]
    else:
        targets = [(factory_col, factory_row + 8)]

    step = bfs_first_step(
        (col, row),
        targets,
        obs,
        config,
        depth=16,
        occupied=occupied,
    )
    if step and step != "IDLE":
        return step

    for direction in ("NORTH", "EAST", "WEST"):
        if has_road(obs, config, col, row, direction):
            next_cell = neighbor(col, row, direction)
            if next_cell not in occupied:
                return direction
    return "IDLE"


def worker_action(
    col: int,
    row: int,
    energy: int,
    factory_pos: tuple[int, int],
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]] | None = None,
) -> str:
    """Open a route ahead of the factory without adding broad economy logic."""
    occupied = occupied or set()
    remove_cost = get_value(
        config,
        "wallRemoveCost",
        DEFAULT_WALL_REMOVE_COST,
    )
    if energy >= remove_cost and not has_road(obs, config, col, row, "NORTH"):
        return "REMOVE_NORTH"

    factory_col, factory_row = factory_pos
    target = (factory_col, factory_row + WORKER_FRONT_OFFSET)
    step = bfs_first_step(
        (col, row),
        [target],
        obs,
        config,
        depth=10,
        occupied=occupied,
    )
    if step and step != "IDLE":
        return step

    for direction in ("NORTH", "EAST", "WEST"):
        if has_road(obs, config, col, row, direction):
            next_cell = neighbor(col, row, direction)
            if next_cell not in occupied:
                return direction
    return "IDLE"


def can_spawn_scout(
    col: int,
    row: int,
    energy: int,
    build_cd: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]],
) -> bool:
    """Return True when the factory can safely spawn one scout north."""
    scout_cost = get_value(config, "scoutCost", DEFAULT_SCOUT_COST)
    spawn = neighbor(col, row, "NORTH")
    south_bound = get_value(obs, "southBound", 0)
    factory_gap = row - south_bound
    danger_gate_ok = (
        not USE_FACTORY_DANGER_GATE
        or factory_gap > FACTORY_DANGER_GAP
    )
    return (
        danger_gate_ok
        and build_cd == 0
        and energy >= scout_cost
        and spawn not in occupied
        and has_road(obs, config, col, row, "NORTH")
    )


def can_spawn_worker(
    col: int,
    row: int,
    energy: int,
    build_cd: int,
    scout_count: int,
    obs: Any,
    config: Any,
    occupied: set[tuple[int, int]],
) -> bool:
    """Return True when the worker experiment can safely spend energy."""
    worker_cost = get_value(config, "workerCost", DEFAULT_WORKER_COST)
    spawn = neighbor(col, row, "NORTH")
    south_bound = get_value(obs, "southBound", 0)
    factory_gap = row - south_bound
    return (
        scout_count > 0
        and factory_gap > WORKER_BUILD_GAP
        and build_cd == 0
        and energy >= max(worker_cost, WORKER_MIN_FACTORY_ENERGY)
        and spawn not in occupied
        and has_road(obs, config, col, row, "NORTH")
    )


def agent_v2(obs: Any, config: Any) -> dict[str, str]:
    """Jump-BFS reference with one scout and one conservative worker."""
    reset_game_state_if_needed(obs)
    update_wall_memory(obs, config)
    actions = {}
    robots = my_robots(obs)
    _, factory_data = my_factory(obs)
    occupied = occupied_positions(robots)
    scout_count = sum(1 for data in robots.values() if data[0] == SCOUT)
    worker_count = sum(1 for data in robots.values() if data[0] == WORKER)

    for uid, data in robots.items():
        robot_type, col, row, energy = data[0], data[1], data[2], data[3]
        move_cd = data[5] if len(data) > 5 else 0
        jump_cd = data[6] if len(data) > 6 else 0
        build_cd = data[7] if len(data) > 7 else 0
        occupied_without_self = occupied - {(col, row)}

        if robot_type == FACTORY:
            if scout_count < MAX_ACTIVE_SCOUTS and can_spawn_scout(
                col,
                row,
                energy,
                build_cd,
                obs,
                config,
                occupied_without_self,
            ):
                actions[uid] = "BUILD_SCOUT"
                scout_count += 1
            elif worker_count < MAX_ACTIVE_WORKERS and can_spawn_worker(
                col,
                row,
                energy,
                build_cd,
                scout_count,
                obs,
                config,
                occupied_without_self,
            ):
                actions[uid] = "BUILD_WORKER"
                worker_count += 1
            else:
                actions[uid] = factory_path_action(
                    col,
                    row,
                    move_cd,
                    jump_cd,
                    obs,
                    config,
                    occupied_without_self,
                )
        elif robot_type == SCOUT and factory_data is not None:
            factory_pos = (factory_data[1], factory_data[2])
            actions[uid] = scout_action(
                col,
                row,
                energy,
                factory_pos,
                obs,
                config,
                occupied_without_self,
            )
        elif robot_type == WORKER and factory_data is not None:
            factory_pos = (factory_data[1], factory_data[2])
            actions[uid] = worker_action(
                col,
                row,
                energy,
                factory_pos,
                obs,
                config,
                occupied_without_self,
            )

    return actions

def compute_actions(obs: Any, config: Any) -> dict[str, str]:
    """Return the production starter action dictionary."""
    return agent_v2(obs, config)


def idle_actions(obs: Any) -> dict[str, str]:
    """Return safe idle actions for visible owned robots."""
    return {uid: "IDLE" for uid in my_robots(obs)}


def agent(obs: Any, config: Any) -> dict[str, str]:
    """Kaggle-facing entry point."""
    try:
        return compute_actions(obs, config)
    except Exception:
        return idle_actions(obs)


def act(obs: Any, config: Any) -> dict[str, str]:
    """Alias used by some local runners."""
    return agent(obs, config)


__all__ = ["agent", "act", "compute_actions"]


## 9. Verify Generated Files

The generated files should compile and stay byte-for-byte identical. Keep this check close to the write cell so stale submission artifacts are obvious.


In [ ]:
from pathlib import Path
import py_compile

main_path = Path("main.py")
submission_path = Path("submission.py")
submission_path.write_text(
    main_path.read_text(encoding="utf-8"),
    encoding="utf-8",
)

py_compile.compile(str(main_path), doraise=True)
py_compile.compile(str(submission_path), doraise=True)
assert main_path.read_text(encoding="utf-8") == submission_path.read_text(
    encoding="utf-8"
)

print("main.py: OK")
print("submission.py: OK")
print("submission.py sync: OK")


## 10. Batch Evaluation

Optional multi-seed evaluation for local signal. Keep `RUN_BATCH_EVALUATION = False` for normal Kaggle submission runs, then enable it when comparing policy variants.


In [ ]:
def summarize_match(agent_fn, seed: int) -> dict[str, int | str]:
    """Run one random-opponent match and summarize key signals."""
    global _last_south_bound
    _wall_memory.clear()
    _last_south_bound = None

    env = make("crawl", configuration={"randomSeed": seed}, debug=True)
    env.run([agent_fn, "random"])

    counts: dict[str, int] = {}
    min_gap = 999
    for step in env.steps:
        state = step[0]
        for action in (state.action or {}).values():
            counts[action] = counts.get(action, 0) + 1

        obs = state.observation
        robots = getattr(obs, "robots", {}) or {}
        player = getattr(obs, "player", 0)
        south_bound = getattr(obs, "southBound", 0)
        for data in robots.values():
            if len(data) > 4 and data[4] == player and data[0] == FACTORY:
                min_gap = min(min_gap, data[2] - south_bound)

    final = env.steps[-1][0]
    return {
        "seed": seed,
        "reward": final.reward,
        "status": final.status,
        "steps": len(env.steps) - 1,
        "build_scout": counts.get("BUILD_SCOUT", 0),
        "build_worker": counts.get("BUILD_WORKER", 0),
        "removes": sum(
            value for key, value in counts.items() if key.startswith("REMOVE_")
        ),
        "transfers": sum(
            value for key, value in counts.items() if key.startswith("TRANSFER_")
        ),
        "jumps": sum(
            value for key, value in counts.items() if key.startswith("JUMP_")
        ),
        "idle": counts.get("IDLE", 0),
        "min_factory_gap": min_gap,
    }


if RUN_SIMULATIONS and RUN_BATCH_EVALUATION:
    eval_rows = []
    for seed in range(EVAL_SEEDS):
        eval_rows.append({"agent": "v1", **summarize_match(agent_v1, seed)})
        eval_rows.append({"agent": "v2", **summarize_match(agent_v2, seed)})
    for row in eval_rows:
        print(row)


## 11. Submit To The Leaderboard

After running the notebook on Kaggle, use **Save Version** and then **Submit to Competition**. The generated `main.py` is the file Kaggle evaluates.


## 12. Next Improvements

This notebook tests whether worker wall removal can improve the Version 6 jump-BFS reference. Useful next experiments:

- Compare worker agent against the Version 6 reference on Kaggle before promoting it.
- Add action-count diagnostics for `BUILD_WORKER` and `REMOVE_*`.
- Tune `WORKER_BUILD_GAP` and `WORKER_MIN_FACTORY_ENERGY` only after seeing replay failures.
- Try miners only after the worker policy improves survival or energy conversion.
